In [ ]:
import sys
import oracledb

INSTANT_CLIENT_DIR = r"C:\Users\E1445612\oracle64client"

try:
    oracledb.init_oracle_client(lib_dir=INSTANT_CLIENT_DIR)
except Exception as e:
    print(f"[-] Client Init Failed: {e}")
    sys.exit(1)

DB_USER = "apps"
DB_PASS = "Sukh#na7"
DB_DSN = "BU2EBS.EMRSN.COM:34101/BETSYU2"

def fetch_top_5_invoices():
    print("[+] Connecting to Oracle EBS...")
    try:
        connection = oracledb.connect(user=DB_USER, password=DB_PASS, dsn=DB_DSN)
        cursor = connection.cursor()
        print("[+] Connected successfully.\n")

        # Force execution through the primary key index (INVOICE_ID) to bypass VPD lock scans
        query = """
        SELECT invoice_id, invoice_num, vendor_id, invoice_amount, invoice_date
          FROM apps.ap_invoices_all
         WHERE invoice_id IN (
             SELECT invoice_id 
               FROM (
                   SELECT invoice_id 
                     FROM apps.ap_invoices_all 
                    ORDER BY invoice_id DESC
               ) 
              WHERE ROWNUM <= 5
         )
        """

        print("[+] Executing indexed query on AP_INVOICES_ALL...")
        cursor.execute(query)
        rows = cursor.fetchall()

        print(f"\n[+] Successfully retrieved {len(rows)} records:")
        print("-" * 80)
        print(f"{'INVOICE_ID':<12} | {'INVOICE_NUM':<25} | {'VENDOR_ID':<10} | {'AMOUNT':<12}")
        print("-" * 80)
        for r in rows:
            print(f"{r[0]:<12} | {str(r[1]):<25} | {str(r[2]):<10} | {str(r[3]):<12}")
        print("-" * 80)

        cursor.close()
        connection.close()

    except Exception as e:
        print(f"[-] Query Failed: {e}")

if __name__ == "__main__":
    fetch_top_5_invoices()